This notebook uses the processed outputs from Notebook 01 to select candidate pairs and build spread/z-score features for pairs trading research.

The workflow is:

1. Load the validated adjusted close price matrix.
2. Load the dollar-volume matrix created from adjusted close and adjusted volume.
3. Run systematic pair selection using correlation, cointegration, half-life, and liquidity filters.
4. Select the top candidate pairs.
5. Estimate hedge ratios and intercepts for selected pairs.
6. Compute residual spreads for the selected pairs.
7. Compute rolling z-scores for mean-reversion signal analysis.

This notebook does not perform backtesting or show performance metrics.That is the scope of Notebook 03.

## 1. Setup and Imports

This section sets the project root, configures Python imports, and loads the required libraries.

The project root is detected so that the notebook can access files from the `src/` and `data/processed/` directories consistently, regardless of whether the notebook is launched from the project root or from the `notebooks/` folder.

In [1]:
import os
import sys
from pathlib import Path
import json
import pickle

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))
    
from src.pairs_selection import select_top_pairs
from src.spread_model import compute_log_prices, estimate_hedge_model, compute_spread, compute_zscore
from src.config_loader import (
    load_run_config,
    load_universe_config
)
from src.walk_forward import (
    generate_walk_forward_folds,
    get_fold_data
)

In [2]:
print("Project root:", PROJECT_ROOT)
print("Current working directory:", Path.cwd())

Project root: c:\Users\Aditi\Aditi_Workspaces\VSCode\QuantProjects\pairs-trading-ai-tech
Current working directory: c:\Users\Aditi\Aditi_Workspaces\VSCode\QuantProjects\pairs-trading-ai-tech\notebooks


## 2. Load Processed Data

This section loads the processed matrices created in Notebook 01:

`price_matrix.csv`: validated adjusted close prices by date and ticker
`dollar_volume_matrix.csv`: validated adjusted close multiplied by validated adjusted volume

The dollar-volume matrix is used as a liquidity input for pair selection. This helps avoid selecting pairs that look statistically attractive but not realistically tradeable.

In [21]:
run_json_file_path = PROJECT_ROOT/"config/run_config.json"
v2_universe_file_path = PROJECT_ROOT/"config/v2_universe.json"

config = load_run_config(run_json_file_path)
v2_univ = load_universe_config(v2_universe_file_path)
    
start_date = config["date_range"]["start_date"]
end_date = config["date_range"]["end_date"]
start_year = pd.Timestamp(start_date).strftime('%Y')
end_year = pd.Timestamp(end_date).strftime('%Y')
processed_file_path = config["data_files"]["processed_dir"]
price_file_name = config["data_files"]["price_matrix"]
dollar_volume_file_name = config["data_files"]["dollar_volume_matrix"]
univ_code = v2_univ["code"]
extension = config["data_files"]["csv_extension"]
price_file_path_name = f"{price_file_name}{univ_code}{start_year}_{end_year}{extension}"
dollar_volume_file_path_name = f"{dollar_volume_file_name}{univ_code}{start_year}_{end_year}{extension}"

price_file_path = PROJECT_ROOT/processed_file_path/price_file_path_name
dollar_volume_file_path = PROJECT_ROOT/processed_file_path/dollar_volume_file_path_name

price_matrix = pd.read_csv(price_file_path, index_col='date', parse_dates=True)
dollar_volume_matrix = pd.read_csv(dollar_volume_file_path, index_col='date', parse_dates=True)

print("Column alignment check: ",price_matrix.columns.equals(dollar_volume_matrix.columns))
print("Index alignment check: ",price_matrix.index.equals(dollar_volume_matrix.index))

print(price_matrix.shape)
print(dollar_volume_matrix.shape)

Column alignment check:  True
Index alignment check:  True
(1760, 50)
(1760, 50)


## 3. Run Pair Selection

This section applies the pair-selection pipeline to the stock universe.

Candidate pairs are evaluated using:
- log-return correlation
- cointegration p-value
- half-life of mean reversion
- average dollar-volume liquidity

The output is a ranked table of the top candidate pairs.

In [22]:
print("Price columns:")
print(list(price_matrix.columns))

print("\nDollar volume columns:")
print(list(dollar_volume_matrix.columns))

print("\nColumns in price but not dollar volume:")
print(set(price_matrix.columns) - set(dollar_volume_matrix.columns))

print("\nColumns in dollar volume but not price:")
print(set(dollar_volume_matrix.columns) - set(price_matrix.columns))

print(price_matrix.shape)
print(dollar_volume_matrix.dtypes)

Price columns:
['MSFT', 'AAPL', 'AMZN', 'GOOGL', 'BRK-B', 'META', 'JNJ', 'JPM', 'V', 'XOM', 'WMT', 'BAC', 'PG', 'PFE', 'UNH', 'VZ', 'T', 'CVX', 'INTC', 'WFC', 'KO', 'CSCO', 'MA', 'HD', 'MRK', 'ORCL', 'BA', 'CMCSA', 'PEP', 'DIS', 'ABBV', 'MCD', 'AMGN', 'ABT', 'PM', 'MDT', 'NFLX', 'NKE', 'CRM', 'IBM', 'UNP', 'TXN', 'HON', 'LLY', 'ACN', 'COST', 'AVGO', 'NVDA', 'TMO', 'BMY']

Dollar volume columns:
['MSFT', 'AAPL', 'AMZN', 'GOOGL', 'BRK-B', 'META', 'JNJ', 'JPM', 'V', 'XOM', 'WMT', 'BAC', 'PG', 'PFE', 'UNH', 'VZ', 'T', 'CVX', 'INTC', 'WFC', 'KO', 'CSCO', 'MA', 'HD', 'MRK', 'ORCL', 'BA', 'CMCSA', 'PEP', 'DIS', 'ABBV', 'MCD', 'AMGN', 'ABT', 'PM', 'MDT', 'NFLX', 'NKE', 'CRM', 'IBM', 'UNP', 'TXN', 'HON', 'LLY', 'ACN', 'COST', 'AVGO', 'NVDA', 'TMO', 'BMY']

Columns in price but not dollar volume:
set()

Columns in dollar volume but not price:
set()
(1760, 50)
MSFT     float64
AAPL     float64
AMZN     float64
GOOGL    float64
BRK-B    float64
META     float64
JNJ      float64
JPM      float64
V 

In [23]:
folds = generate_walk_forward_folds()
print(folds)


[{'fold': 1, 'train': ('2019-01-01', '2021-12-31'), 'test': ('2022-01-01', '2022-12-31'), 'oos': ('2023-01-01', '2023-12-31')}, {'fold': 2, 'train': ('2020-01-01', '2022-12-31'), 'test': ('2023-01-01', '2023-12-31'), 'oos': ('2024-01-01', '2024-12-31')}, {'fold': 3, 'train': ('2021-01-01', '2023-12-31'), 'test': ('2024-01-01', '2024-12-31'), 'oos': ('2025-01-01', '2025-12-31')}]


In [27]:
train_price,test_price,oos_price = get_fold_data(price_matrix, folds[0])
print(train_price.index.min(),test_price.index.min(),oos_price.index.min())
train_dv, test_dv, oos_dv = get_fold_data(dollar_volume_matrix, folds[0])

2019-01-02 00:00:00+00:00 2022-01-03 00:00:00+00:00 2023-01-03 00:00:00+00:00


In [28]:
pair_config = config["pairs_selection"]

top_pairs = select_top_pairs(
    train_price,
    train_dv,
    top_n = pair_config["top_n"],
    min_correlation = pair_config["min_correlation"],
    pvalue_threshold = pair_config["pvalue_threshold"],
    trend = pair_config["trend"],
    use_log_prices = pair_config["use_log_prices"],
    max_half_life = pair_config["max_half_life"],
    min_avg_dollar_volume = pair_config["min_avg_dollar_volume"],
    min_observations = pair_config["min_observations"]
)

top_pairs

,asset_y,asset_x,correlation,cointegration_pvalue,cointegration_passed,half_life,liquidity_passed,avg_dollar_volume_y,avg_dollar_volume_x,rank
0,PEP,COST,0.622783,0.000248,True,9.566572,True,5.342328e+08,7.227777e+08,1
1,UNP,AVGO,0.614099,0.000360,True,9.769209,True,5.174857e+08,7.066761e+08,2
2,CMCSA,HON,0.583763,0.000571,True,10.033131,True,7.297389e+08,4.687945e+08,3
3,BRK-B,PM,0.630418,0.001012,True,17.274298,True,1.098229e+09,3.200314e+08,4
4,JNJ,ABBV,0.456989,0.002040,True,13.286042,True,9.272477e+08,5.844794e+08,5
5,MRK,ACN,0.507550,0.002762,True,14.177467,True,7.178727e+08,4.324241e+08,6
6,MRK,ORCL,0.400743,0.002926,True,14.937243,True,7.178727e+08,7.180879e+08,7
7,CRM,TMO,0.460507,0.003301,True,17.707300,True,1.280174e+09,5.983532e+08,8
8,MRK,MCD,0.469072,0.003373,True,14.310966,True,7.178727e+08,5.917228e+08,9
9,MRK,TMO,0.537248,0.003641,True,14.877909,True,7.178727e+08,5.983532e+08,10


### Model Assumptions for Pair Selection

The pair-selection process uses configurable default assumptions. These are chosen for a simple v1 daily-data research pipeline and should be stress-tested in later versions.

- **Universe:** The initial universe uses large-cap technology stocks to prioritize liquidity, data quality, and clean first-pass testing. This introduces sector concentration and possible survivorship bias.
- **Correlation:** Pair correlation is computed on log returns rather than raw price levels, because raw prices may trend together even when short-term co-movement is weak.
- **Cointegration:** The cointegration test uses log-price levels with a default p-value threshold of 0.05. This is a standard first-pass significance threshold, but later versions should test stricter cutoffs and multiple-testing corrections.
- **Trend assumption:** The cointegration test uses `trend="c"`, meaning a constant/intercept is allowed, but no deterministic linear trend is assumed.
- **Half-life:** Half-life is estimated using a one-observation lag because the data frequency is daily. This gives a first-order estimate of mean-reversion speed.
- **Liquidity:** Liquidity is filtered using average dollar volume, calculated as adjusted close times adjusted volume. This is preferred over raw share volume because it accounts for stock price.
- **Top N:** The notebook selects the top 5 pairs to avoid manual cherry-picking while keeping the analysis readable.

## 4. Inspect Selected Pairs

This section reviews the selected top pairs and checks whether the results are economically and statistically reasonable.

The ranking prioritizes:
1. lower cointegration p-values,
2. higher return correlation,
3. lower half-life,
4. liquidity passing the dollar-volume filter.

In [ ]:
selected_pairs = top_pairs.copy()

print(selected_pairs.iloc[0:10])

top_pair = selected_pairs.iloc[0]
asset_y = top_pair["asset_y"]
asset_x = top_pair["asset_x"]

  asset_y asset_x  correlation  cointegration_pvalue  cointegration_passed  \
0     PEP    COST     0.622783              0.000248                  True   
1     UNP    AVGO     0.614099              0.000360                  True   
2   CMCSA     HON     0.583763              0.000571                  True   
3   BRK-B      PM     0.630418              0.001012                  True   
4     JNJ    ABBV     0.456989              0.002040                  True   
5     MRK     ACN     0.507550              0.002762                  True   
6     MRK    ORCL     0.400743              0.002926                  True   
7     CRM     TMO     0.460507              0.003301                  True   
8     MRK     MCD     0.469072              0.003373                  True   
9     MRK     TMO     0.537248              0.003641                  True   

   half_life  liquidity_passed  avg_dollar_volume_y  avg_dollar_volume_x  rank  
0   9.566572              True         5.342328e+08         

AttributeError: 'DataFrame' object has no attribute 'sharpe'

The pipeline was configured to return up to 5 pairs. Under the stricter 5% cointegration threshold, 2 pairs passed all filters. These pairs are carried forward into spread modeling rather than forcing weaker candidates into the analysis.

## 5. Build Spread and Z-Score for the Top Pair

This section uses the highest-ranked pair to demonstrate the spread modeling workflow.

The process is:
1. compute log prices,
2. estimate the hedge ratio and intercept,
3. compute the residual spread,
4. compute the rolling z-score.

The z-score measures how far the current spread is from its recent rolling mean in units of rolling standard deviation.

In [ ]:
min_observations = config["pairs_selection"]["min_observations"]
log_price_matrix = compute_log_prices(price_matrix)
top_pair_hedge_model = estimate_hedge_model(log_price_matrix, asset_y, asset_x, min_observations=min_observations)
print("\n Hedge model:",top_pair_hedge_model)
top_pair_spread = compute_spread(log_price_matrix, top_pair_hedge_model)
print("\n Spread head:",top_pair_spread.head())
print("\n Spread summary:",top_pair_spread.describe())
window = config["spread_model"]["z_score_window"]
top_pair_zscore = compute_zscore(top_pair_spread,window=window)
print("\n Z-score head:",top_pair_zscore.head())
print("\n Z-score summary:",top_pair_zscore.describe())

In [ ]:
top_pair_spread.plot(figsize=(12,6))
plt.title("Spread over time")
plt.xlabel("Dates")
plt.ylabel("Spread")
plt.legend(title="Top Pair")
plt.grid(True)
plt.show()

In [ ]:
top_pair_zscore.plot(figsize=(12,6))
plt.title("Zscore over time")
plt.xlabel("Dates")
plt.ylabel("Zscore")

plt.axhline(y=2, color='r', linestyle='--', label='Threshold')
plt.axhline(y=0, color='g', linestyle='--', label='Threshold')
plt.axhline(y=-2, color='r', linestyle='--', label='Threshold')
plt.legend(title="Top Pair")
plt.grid(True)
plt.show()

## 6. Extend Spread Modeling to all the selected pairs

This section applies the spread and z-score workflow to all selected top pairs.

The goal is to prepare multiple candidate spreads for later backtesting rather than relying on a single manually selected pair.

In [ ]:
#Reusing log_price_matrix
hedge_models_by_pair = {}
spreads_by_pair = {}
zscores_by_pair = {}
summary_rows = []

for _, pair in selected_pairs.iterrows():
    asset_y = pair["asset_y"]
    asset_x = pair["asset_x"]
    pair_name = f"{asset_y}_{asset_x}"
    hedge_model = estimate_hedge_model(log_price_matrix, asset_y, asset_x, min_observations=min_observations)
    spread = compute_spread(log_price_matrix,hedge_model)
    zscore = compute_zscore(spread,window=window)
    
    hedge_models_by_pair[pair_name] = hedge_model
    spreads_by_pair[pair_name] = spread
    zscores_by_pair[pair_name] = zscore
    
    summary_rows.append({
        "pair_name":pair_name,
        "asset_y":asset_y,
        "asset_x":asset_x,
        "alpha":hedge_model["alpha"],
        "beta":hedge_model["beta"],
        "spread_mean":spread.mean(),
        "spread_std":spread.std(),
        "zscore_mean":zscore.mean(),
        "zscore_std":zscore.std(),
        "zscore_min":zscore.min(),
        "zscore_max":zscore.max(),
        "num_observations":spread.dropna().shape[0]
    })
    
pair_model_summary_df = pd.DataFrame(summary_rows)
pair_model_summary_df

print(hedge_models_by_pair.keys())
print(spreads_by_pair.keys())
print(zscores_by_pair.keys())

## 7. Notebook Summary

This notebook completed the pair-selection and spread-modeling stage of the pairs trading research pipeline.

Using the processed price and dollar-volume matrices from Notebook 01, the notebook:

1. loaded run settings from `run_config.json`,
2. loaded validated adjusted close and dollar-volume data,
3. selected candidate pairs using correlation, cointegration, half-life, and liquidity filters,
4. inspected the selected pairs,
5. estimated hedge models for the selected pairs,
6. computed residual spreads,
7. computed rolling z-scores for signal analysis.

The pair-selection pipeline was configured to return up to 5 pairs. Under the stricter 5% cointegration threshold, 2 pairs passed all configured filters. These pairs were carried forward into spread and z-score modeling rather than forcing weaker candidates into the analysis.

Key outputs from this notebook:

- `selected_pairs`: ranked pairs that passed the configured filters
- `hedge_models_by_pair`: hedge model parameters for each selected pair
- `spreads_by_pair`: residual spread series for each selected pair
- `zscores_by_pair`: rolling z-score series for each selected pair
- `pair_model_summary_df`: compact summary of hedge model and spread/z-score diagnostics

These outputs will be used in the next stage of the project: backtesting.

The backtesting module will convert z-score signals into positions, calculate strategy returns, and evaluate performance using metrics such as Sharpe ratio, drawdown, alpha, and Newey-West adjusted t-statistics.

In [ ]:
print("Notebook 02 complete.")
print("Number of selected pairs:", len(selected_pairs))
print("Stored spreads:", list(spreads_by_pair.keys()))
print("Stored z-scores:", list(zscores_by_pair.keys()))

pair_model_summary_df

## 8. Save Pair Selection and Spread Model Outputs

This section saves the outputs needed by Notebook 03 for backtesting and performance analysis.

Saved artifacts:

- selected pair table
- hedge model dictionary
- spread dictionary
- z-score dictionary
- lightweight run metadata

Pickle is used for Python objects such as dictionaries of pandas Series.
CSV is used for the selected pair table.
JSON is used for human-readable metadata.

In [ ]:
print(type(selected_pairs))
print(type(hedge_models_by_pair))
print(type(spreads_by_pair))
print(type(zscores_by_pair))
print(type(pair_model_summary_df))

In [ ]:
# Create outputs folder
output_dir = PROJECT_ROOT/"outputs"
output_dir.mkdir(parents=True, exist_ok=True)

# Save readable CSV outputs
selected_pairs.to_csv(output_dir / "v1_selected_pairs.csv", index=False)
pair_model_summary_df.to_csv(output_dir / "v1_pair_model_summary.csv", index=False)

# Bundle Python objects for Notebook 03
pair_model_outputs = {
    "selected_pairs": selected_pairs,
    "hedge_models_by_pair": hedge_models_by_pair,
    "spreads_by_pair": spreads_by_pair,
    "zscores_by_pair": zscores_by_pair,
    "pair_model_summary_df": pair_model_summary_df,
}

# Save pickle bundle
with open(output_dir / "v1_pair_model_outputs.pkl", "wb") as f:
    pickle.dump(pair_model_outputs, f)

# Save lightweight JSON metadata
metadata = {
    "artifact_name": "v1_pair_model_outputs",
    "notebook": "02_spread_modeling",
    "selected_pair_count": len(selected_pairs),
    "selected_pair_names": list(hedge_models_by_pair.keys()),
    "saved_files": {
        "selected_pairs_csv": "outputs/v1_selected_pairs.csv",
        "pair_model_summary_csv": "outputs/v1_pair_model_summary.csv",
        "pair_model_outputs_pickle": "outputs/v1_pair_model_outputs.pkl",
    },
}

with open(output_dir / "v1_pair_model_metadata.json", "w") as f:
    json.dump(metadata, f, indent=4)

print("Saved Notebook 02 outputs for Notebook 03.")
print(f"Selected pairs saved: {len(selected_pairs)}")
print(f"Pair model outputs saved to: {output_dir / 'v1_pair_model_outputs.pkl'}")

In [ ]:
list(output_dir.glob("v1_*"))

In [ ]:
# Save pickle bundle
with open(output_dir / "v1_pair_model_outputs.pkl", "rb") as f:
    pair_model_outputs = pickle.load(f)

In [ ]:
print(pair_model_outputs["zscores_by_pair"]['MA_NVDA'].min())